# YOLO detection - full training run on a GPU

The local CPU run was compute-limited: **yolov8n, 10 epochs, 416 px**, reaching
mAP@50 0.602 with the metric still climbing on the final epoch. It stopped because
each epoch took 5-7 minutes on CPU, not because the model had finished learning.

This notebook does the run properly on a T4: **yolov8s, 50 epochs, 640 px**, about
an hour. It runs the repo's own `src/train_yolo.py` and `src/export_yolo.py` rather
than reimplementing them, so the result is reproducible locally.

At the end it exports ONNX for the web demo, so the deployed detector can be
upgraded from the same run.

**Runtime → Change runtime type → T4 GPU** before starting.

## 1. Confirm the GPU

In [ ]:
import torch

print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run. "
                     "On CPU this would take days.")

## 2. Mount Drive and install Ultralytics

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxscript

## 3. Settings

`yolov8s` rather than the `yolov8n` used locally. Worth knowing for the write-up:
this run changes three variables at once (model size, image size, epochs), so it is
a better *result* but not a clean ablation of any one factor. Set `MODEL` to
`yolov8n.pt` if you want to isolate the effect of epochs and resolution alone.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/cardd'
PROJECT   = '/content/AI-Image-Processing'

MODEL  = 'yolov8s.pt'    # 'yolov8n.pt' to match the local run's architecture
EPOCHS = 50
IMGSZ  = 640

## 4. Unpack the repo source

In [ ]:
import os, zipfile
from pathlib import Path

assert Path(f'{DRIVE_DIR}/cardd_src.zip').exists(), (
    f'cardd_src.zip not found in {DRIVE_DIR}. Zip src/, docs/, requirements*.txt '
    'and README.md, then upload it there.')

os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(f'{DRIVE_DIR}/cardd_src.zip') as zf:
    zf.extractall(PROJECT)
os.chdir(PROJECT)
print("source unpacked ->", PROJECT)

## 5. Get the dataset

YOLO trains on the **original** CarDD images and boxes, not the cropped
classification set, so it needs the raw ~3 GB download. Downloading it inside Colab
is far quicker than uploading it.

Needs a Kaggle API token. Either put `kaggle.json` in `MyDrive/cardd/` once, or the
cell below will prompt you to upload it.

In [ ]:
from pathlib import Path
import shutil

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
target = kaggle_dir / 'kaggle.json'

drive_token = Path(DRIVE_DIR) / 'kaggle.json'
if drive_token.exists():
    shutil.copy2(drive_token, target)
    print("using kaggle.json from Drive")
elif not target.exists():
    from google.colab import files
    print("Upload kaggle.json (kaggle.com/settings -> Create New API Token):")
    up = files.upload()
    Path('kaggle.json').replace(target)
    print("uploaded")
else:
    print("kaggle.json already present")

target.chmod(0o600)

In [ ]:
# Skip the download if the raw data is already unpacked in this session.
if Path(f'{PROJECT}/data/raw/cardd/data.yaml').exists():
    print("raw dataset already present - skipping download")
else:
    !python src/download_data.py

In [ ]:
import subprocess
counts = {}
for split in ('train', 'val', 'test'):
    d = Path(f'{PROJECT}/data/raw/cardd/{split}/images')
    counts[split] = len(list(d.glob('*'))) if d.exists() else 0
print("images per split:", counts)
assert counts['train'] > 0, "no training images found - check the download"

## 6. Train

Runs `src/train_yolo.py` unchanged. That script rewrites CarDD's `data.yaml` (which
hardcodes Kaggle paths) for this machine, trains, then runs an explicit validation
pass and writes `results/yolo/metrics.json` in the same shape the classifiers use, so
`src/compare.py` picks it up.

Roughly an hour on a T4. Ultralytics prints per-epoch metrics as it goes.

In [ ]:
!python src/train_yolo.py --data "data/raw/cardd/data.yaml" \
    --epochs {EPOCHS} --img-size {IMGSZ} --weights {MODEL}

## 7. Results

In [ ]:
import json
from IPython.display import Image, display

metrics = json.loads(Path(f'{PROJECT}/results/yolo/metrics.json').read_text())
print("=== this run ===")
for k in ('precision', 'recall', 'mAP50', 'mAP50_95'):
    print(f"  {k:10s}: {metrics[k]:.4f}")

print("\n=== local CPU run, for comparison ===")
print("  precision : 0.6338\n  recall    : 0.5663")
print("  mAP50     : 0.6021\n  mAP50_95  : 0.4869")

gain = metrics['mAP50'] - 0.6021
print(f"\nmAP@50 change: {gain:+.4f}")

In [ ]:
for name in ('results.png', 'confusion_matrix.png', 'BoxPR_curve.png'):
    p = Path(f'{PROJECT}/results/yolo/cardd/{name}')
    if p.exists():
        print(name)
        display(Image(str(p)))

### Did it converge this time?

Check the tail of the training log. If mAP is still rising on the last few epochs the
run is *still* compute-limited and more epochs would help - which is worth stating in
the report either way.

In [ ]:
import csv

rows = list(csv.DictReader(open(f'{PROJECT}/results/yolo/cardd/results.csv')))
key = next(k for k in rows[0] if 'mAP50(B)' in k and '95' not in k)
tail = rows[-6:]
print(f"{'epoch':>6} {'mAP50':>9}")
for r in tail:
    print(f"{r['epoch'].strip():>6} {float(r[key]):9.4f}")

delta = float(tail[-1][key]) - float(tail[0][key])
print(f"\nchange over the last {len(tail)} epochs: {delta:+.4f}")
print("still improving - more epochs would help" if delta > 0.005
      else "plateaued - this is close to what this configuration can do")

## 8. Export ONNX for the web demo

Exports without built-in NMS, because the demo decodes the raw output and runs NMS in
numpy to keep torch out of its bundle. The exported `yolo_meta.json` records the image
size, and the app reads it from there - so switching from 416 to 640 needs no code
change in the demo.

In [ ]:
!python src/export_yolo.py --weights "results/yolo/cardd/weights/best.pt" \
    --img-size {IMGSZ} --out models

## 9. Copy everything to Drive

In [ ]:
import shutil

dest = Path(DRIVE_DIR) / 'yolo_full'
dest.mkdir(parents=True, exist_ok=True)

for rel in ('models/yolo_damage_detector.onnx', 'models/yolo_meta.json',
            'results/yolo/metrics.json',
            'results/yolo/cardd/results.png', 'results/yolo/cardd/results.csv',
            'results/yolo/cardd/confusion_matrix.png',
            'results/yolo/cardd/weights/best.pt'):
    src = Path(PROJECT) / rel
    if src.exists():
        shutil.copy2(src, dest / src.name)
        print(f"saved {src.name:32s} {src.stat().st_size / 1e6:7.1f} MB")

print(f"\nEverything is in Drive at: {dest}")

## Next: upgrade the deployed detector

1. Download `yolo_damage_detector.onnx` and `yolo_meta.json` from
   `MyDrive/cardd/yolo_full/`.
2. Replace the pair in `car-damage-demo/models/`.
3. Push — Vercel redeploys automatically:

```
cd car-damage-demo
git add models && git commit -m "Upgrade detector to the 50-epoch 640px run" && git push
```

Also copy `best.pt` and `results/` into the training repo if you want the run's
artifacts locally; both folders are gitignored.

Then update the mAP figures in the training repo's README, which currently quote the
CPU run.